# Experiment 10 — FT-Transformer

Pure PyTorch FT-Transformer tested on the exact 18 selected Experiment 9 inputs and the common chronological holdout (train before **2024-05**, validate from **2024-05** onward). Every numerical and categorical feature is represented as an attention token. No tree model is used.

In [ ]:
# Shared feature construction and train-only tabular encoding utilities.
"""Experiment 10: leakage-safe, time-aware deep learning for rental listings.

This is deliberately a *pure PyTorch* experiment: no LightGBM, CatBoost, or
other tree model is fitted or blended.  It evaluates the exact 18-column
winning feature set from ``exp_9_target_features.ipynb`` on its common,
forward-looking holdout: rows before 2024-05 fit the model and rows from
2024-05 onward are held out.

The model is a TabTransformer-style regressor.  Self-attention learns
interactions between categorical listing/location tokens, while a dense branch
uses scaled continuous variables (including the chronological approval index
and target-history features).  Category vocabularies, numeric imputation and
scaling, target scaling, preprocessing, and target-history features are all
fit on the early partition only.

Run from the repository root:
    python exploration/metul/exp_10_dl_tabtransformer.py

Set EXP10_EPOCHS, EXP10_BATCH_SIZE, or EXP10_PATIENCE in the environment to
adjust a run without changing the experiment definition.
"""

from __future__ import annotations

import json
import os
import random
import sys
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy.spatial import cKDTree
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from torch import Tensor, nn
from torch.utils.data import DataLoader, TensorDataset


_candidate_roots = [Path.cwd(), *Path.cwd().parents]
if "__file__" in globals():
    _candidate_roots.insert(0, Path(__file__).resolve().parents[2])
PROJECT_ROOT = next(
    (path for path in _candidate_roots if (path / "exploration/metul/preprocessing.py").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run from the project directory or one of its subdirectories.")
EXPERIMENT_DIR = PROJECT_ROOT / "exploration" / "metul"
sys.path.insert(0, str(EXPERIMENT_DIR))

from exp_9_feature_engineering import (  # noqa: E402
    COMPACT_FEATURES,
    GROUP_COLUMNS,
    TARGET,
    TargetFeatureTransformer,
    labelled_history,
    preprocess_partition,
)
from preprocessing import RentalPreprocessor, load_datasets  # noqa: E402


SEED = 42
LAG_MONTHS = 24
EPOCHS = 30
BATCH_SIZE = int(os.environ.get("EXP10_BATCH_SIZE", "1024"))
PATIENCE = int(os.environ.get("EXP10_PATIENCE", "7"))
LEARNING_RATE = float(os.environ.get("EXP10_LEARNING_RATE", "0.0003"))
WEIGHT_DECAY = float(os.environ.get("EXP10_WEIGHT_DECAY", "0.0001"))

# Exact selected Exp. 9 global-lag feature set, in its documented order.
FEATURES = COMPACT_FEATURES + [
    "monthly_target_lag",
    "town_flat_type_target_mean",
    "nearest_mrt_name",
    "nearest_tourist_attraction_name",
    "nearest_monument_name",
]
CATEGORICAL_FEATURES = [
    "TOWN", "BLOCK", "STREET", "FLAT_TYPE", "FLAT_MODEL",
    "nearest_mrt_name", "nearest_tourist_attraction_name", "nearest_monument_name",
]
NUMERIC_FEATURES = [column for column in FEATURES if column not in CATEGORICAL_FEATURES]


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)


def nearest_geojson_name(
    base_frame: pd.DataFrame, filename: str, name_field: str, feature_name: str
) -> pd.Series:
    """Return the nearest supplied geographic point name for each listing."""
    path = PROJECT_ROOT / "data" / "possible_additional_features" / filename
    with path.open() as file:
        geojson = json.load(file)
    points = pd.DataFrame([
        {
            "longitude": feature["geometry"]["coordinates"][0],
            "latitude": feature["geometry"]["coordinates"][1],
            feature_name: feature["properties"][name_field],
        }
        for feature in geojson["features"]
        if feature.get("geometry", {}).get("type") == "Point"
    ]).dropna(subset=["longitude", "latitude", feature_name])
    point_coordinates = points[["longitude", "latitude"]].to_numpy() * [110.96, 111.0]
    row_coordinates = base_frame[["LONGITUDE", "LATITUDE"]].to_numpy() * [110.96, 111.0]
    _, indices = cKDTree(point_coordinates).query(row_coordinates, k=1)
    return pd.Series(
        points[feature_name].astype(str).to_numpy()[indices], index=base_frame.index, name=feature_name
    )


def nearest_mrt_name(base_frame: pd.DataFrame, mrt: pd.DataFrame) -> pd.Series:
    stations = mrt[["NAME", "LATITUDE", "LONGITUDE"]].dropna().copy()
    station_coordinates = stations[["LATITUDE", "LONGITUDE"]].to_numpy() * [111.0, 110.96]
    row_coordinates = base_frame[["LATITUDE", "LONGITUDE"]].to_numpy() * [111.0, 110.96]
    _, indices = cKDTree(station_coordinates).query(row_coordinates, k=1)
    return pd.Series(
        stations["NAME"].astype(str).to_numpy()[indices], index=base_frame.index, name="nearest_mrt_name"
    )


def model_frame(
    base: pd.DataFrame, target_features: pd.DataFrame, datasets: dict[str, pd.DataFrame]
) -> pd.DataFrame:
    """Make the exact Exp. 9 selected-frame without looking at any labels."""
    return pd.concat(
        [
            base[COMPACT_FEATURES].reset_index(drop=True),
            target_features.reset_index(drop=True),
            nearest_mrt_name(base, datasets["mrt"]).reset_index(drop=True),
            nearest_geojson_name(
                base, "TouristAttractions.geojson", "PAGETITLE", "nearest_tourist_attraction_name"
            ).reset_index(drop=True),
            nearest_geojson_name(
                base, "MonumentsGEOJSON.geojson", "NAME", "nearest_monument_name"
            ).reset_index(drop=True),
        ],
        axis=1,
    )[FEATURES]


@dataclass
class TabularEncoder:
    """Train-only categorical vocabularies and numeric robust scaling."""

    categorical_columns: list[str]
    numeric_columns: list[str]

    def fit(self, frame: pd.DataFrame) -> "TabularEncoder":
        self.vocabularies = {}
        for column in self.categorical_columns:
            values = frame[column].fillna("__MISSING__").astype(str)
            # Index 0 is reserved for missing/unseen categories.
            self.vocabularies[column] = {value: index + 1 for index, value in enumerate(pd.unique(values))}
        numeric = frame[self.numeric_columns].apply(pd.to_numeric, errors="coerce")
        self.numeric_medians = numeric.median()
        filled = numeric.fillna(self.numeric_medians)
        self.numeric_means = filled.mean()
        self.numeric_stds = filled.std().replace(0, 1.0).fillna(1.0)
        return self

    @property
    def cardinalities(self) -> list[int]:
        return [len(self.vocabularies[column]) + 1 for column in self.categorical_columns]

    def transform(self, frame: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
        categorical = np.column_stack([
            frame[column].fillna("__MISSING__").astype(str).map(self.vocabularies[column]).fillna(0).to_numpy(dtype=np.int64)
            for column in self.categorical_columns
        ])
        numeric = frame[self.numeric_columns].apply(pd.to_numeric, errors="coerce")
        numeric = numeric.fillna(self.numeric_medians)
        numeric = ((numeric - self.numeric_means) / self.numeric_stds).to_numpy(dtype=np.float32)
        return categorical, numeric


class TabTransformerRegressor(nn.Module):
    """Categorical self-attention, numeric interactions, and an extrapolating time skip."""

    def __init__(
        self, cardinalities: list[int], numeric_count: int, time_index: int, token_dim: int = 32
    ) -> None:
        super().__init__()
        self.embeddings = nn.ModuleList([nn.Embedding(size, token_dim) for size in cardinalities])
        self.column_embedding = nn.Parameter(torch.empty(1, len(cardinalities), token_dim))
        nn.init.normal_(self.column_embedding, std=0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=token_dim, nhead=4, dim_feedforward=token_dim * 3,
            dropout=0.10, activation="gelu", batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=2)
        # A linear neural skip on chronological month makes the learned global
        # market trend explicit and able to extend beyond the fit-period range.
        self.time_index = time_index
        self.time_trend = nn.Linear(1, 1)
        nn.init.zeros_(self.time_trend.weight)
        nn.init.zeros_(self.time_trend.bias)
        dense_input = len(cardinalities) * token_dim + numeric_count
        self.head = nn.Sequential(
            nn.LayerNorm(dense_input),
            nn.Linear(dense_input, 256), nn.GELU(), nn.Dropout(0.15),
            nn.Linear(256, 96), nn.GELU(), nn.Dropout(0.05),
            nn.Linear(96, 1),
        )

    def forward(self, categorical: Tensor, numeric: Tensor) -> Tensor:
        tokens = torch.stack([embedding(categorical[:, i]) for i, embedding in enumerate(self.embeddings)], dim=1)
        encoded = self.transformer(tokens + self.column_embedding).flatten(start_dim=1)
        listing_adjustment = self.head(torch.cat([encoded, numeric], dim=1)).squeeze(1)
        market_trend = self.time_trend(numeric[:, self.time_index:self.time_index + 1]).squeeze(1)
        return listing_adjustment + market_trend


def predict(model: nn.Module, categorical: np.ndarray, numeric: np.ndarray, target_mean: float, target_std: float) -> np.ndarray:
    model.eval()
    batches = DataLoader(TensorDataset(torch.from_numpy(categorical), torch.from_numpy(numeric)), batch_size=BATCH_SIZE)
    outputs: list[Tensor] = []
    with torch.inference_mode():
        for cat_batch, num_batch in batches:
            outputs.append(model(cat_batch, num_batch).cpu())
    return np.expm1(torch.cat(outputs).numpy() * target_std + target_mean).clip(min=0)


def main() -> None:
    seed_everything(SEED)
    torch.set_num_threads(max(1, min(8, os.cpu_count() or 1)))

    _, datasets = load_datasets(PROJECT_ROOT / "data")
    raw_train = datasets["train"].copy()
    raw_train["period"] = pd.to_datetime(raw_train["RENT_APPROVAL_DATE"] + "-01").dt.to_period("M")
    periods = np.sort(raw_train["period"].unique())
    validation_cutoff = periods[int(len(periods) * 0.80)]
    raw_fit = raw_train.loc[raw_train["period"] < validation_cutoff].copy()
    raw_valid = raw_train.loc[raw_train["period"] >= validation_cutoff].copy()
    assert str(validation_cutoff) == "2024-05", f"Unexpected chronological cutoff: {validation_cutoff}"

    preprocessor = RentalPreprocessor(datasets)
    fit_base = preprocess_partition(preprocessor, raw_fit, fit=True)
    valid_base = preprocess_partition(preprocessor, raw_valid)
    target_transformer = TargetFeatureTransformer(LAG_MONTHS)
    fit_targets = target_transformer.fit_transform_train(labelled_history(raw_fit, fit_base))
    valid_targets = target_transformer.transform(labelled_history(raw_valid, valid_base))
    fit_frame = model_frame(fit_base, fit_targets, datasets)
    valid_frame = model_frame(valid_base, valid_targets, datasets)
    assert fit_frame.columns.tolist() == FEATURES == valid_frame.columns.tolist()

    encoder = TabularEncoder(CATEGORICAL_FEATURES, NUMERIC_FEATURES).fit(fit_frame)
    fit_cat, fit_num = encoder.transform(fit_frame)
    valid_cat, valid_num = encoder.transform(valid_frame)
    y_fit = raw_fit[TARGET].to_numpy(dtype=np.float32)
    y_valid = raw_valid[TARGET].to_numpy(dtype=np.float32)
    # Log rent gives the neural loss a less outlier-dominated regression scale;
    # metrics below remain in the original dollar-per-month units.
    log_y_fit = np.log1p(y_fit)
    target_mean, target_std = float(log_y_fit.mean()), float(log_y_fit.std())
    y_fit_scaled = (log_y_fit - target_mean) / target_std

    model = TabTransformerRegressor(
        encoder.cardinalities, len(NUMERIC_FEATURES), NUMERIC_FEATURES.index("approval_date_num")
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)
    loss_function = nn.HuberLoss(delta=1.0)
    train_loader = DataLoader(
        TensorDataset(torch.from_numpy(fit_cat), torch.from_numpy(fit_num), torch.from_numpy(y_fit_scaled)),
        batch_size=BATCH_SIZE, shuffle=True, generator=torch.Generator().manual_seed(SEED),
    )

    best_state: dict[str, Tensor] | None = None
    best_rmse, best_epoch, remaining_patience = float("inf"), 0, PATIENCE
    history = []
    print(f"Exp. 10 | PyTorch TabTransformer | fit={len(raw_fit):,}, valid={len(raw_valid):,}, cutoff={validation_cutoff}")
    print(f"Features ({len(FEATURES)}): {', '.join(FEATURES)}")
    for epoch in range(1, EPOCHS + 1):
        model.train()
        epoch_loss = 0.0
        for cat_batch, num_batch, target_batch in train_loader:
            optimizer.zero_grad(set_to_none=True)
            loss = loss_function(model(cat_batch, num_batch), target_batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item() * len(target_batch)
        prediction = predict(model, valid_cat, valid_num, target_mean, target_std)
        rmse = mean_squared_error(y_valid, prediction) ** 0.5
        scheduler.step(rmse)
        history.append({"epoch": epoch, "train_huber": epoch_loss / len(raw_fit), "validation_rmse": rmse})
        print(f"epoch {epoch:02d}: train Huber={epoch_loss / len(raw_fit):.5f}, validation RMSE={rmse:.3f}")
        if rmse < best_rmse:
            best_rmse, best_epoch, remaining_patience = rmse, epoch, PATIENCE
            best_state = {name: value.detach().clone() for name, value in model.state_dict().items()}
        else:
            remaining_patience -= 1
            if remaining_patience == 0:
                print(f"Early stopping after epoch {epoch}; best epoch was {best_epoch}.")
                break

    assert best_state is not None
    model.load_state_dict(best_state)
    prediction = predict(model, valid_cat, valid_num, target_mean, target_std)
    metrics = {
        "experiment": "exp_10_dl_tabtransformer",
        "framework": "PyTorch",
        "model": "TabTransformer categorical self-attention + numeric MLP + linear neural time-trend skip",
        "tree_models_used": False,
        "validation_protocol": "chronological: train before 2024-05; validate from 2024-05 onward",
        "validation_cutoff": str(validation_cutoff),
        "fit_rows": int(len(raw_fit)),
        "validation_rows": int(len(raw_valid)),
        "lag_months": LAG_MONTHS,
        "features": FEATURES,
        "best_epoch": best_epoch,
        "RMSE": float(mean_squared_error(y_valid, prediction) ** 0.5),
        "MAE": float(mean_absolute_error(y_valid, prediction)),
        "R2": float(r2_score(y_valid, prediction)),
    }
    output_stem = EXPERIMENT_DIR / "exp_10_dl_tabtransformer"
    pd.DataFrame(history).to_csv(output_stem.with_name(output_stem.name + "_history.csv"), index=False)
    pd.DataFrame([metrics]).drop(columns=["features"]).to_csv(
        output_stem.with_name(output_stem.name + "_validation_metrics.csv"), index=False
    )
    output_stem.with_name(output_stem.name + "_validation_metrics.json").write_text(json.dumps(metrics, indent=2))
    print("\nValidation metrics")
    print(pd.Series(metrics)[["RMSE", "MAE", "R2", "best_epoch"]].to_string())
    print(f"Saved metrics and learning curve with prefix: {output_stem.name}_")



In [ ]:
"""Temporary runner; its complete source is copied into the Experiment 10 notebook."""
import json
import os
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

base = sys.modules[__name__]
ROOT = base.PROJECT_ROOT
from preprocessing import RentalPreprocessor, load_datasets

SEED, EPOCHS, BATCH_SIZE, PATIENCE = 42, 30, 4096, 7
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.set_num_threads(max(1, min(8, os.cpu_count() or 1)))


class FTTransformerRegressor(nn.Module):
    """FT-Transformer: every categorical and numerical feature is an attention token."""

    def __init__(self, cardinalities, numeric_count, token_dim=16):
        super().__init__()
        self.categorical_embeddings = nn.ModuleList([nn.Embedding(size, token_dim) for size in cardinalities])
        self.numeric_weight = nn.Parameter(torch.empty(numeric_count, token_dim))
        self.numeric_bias = nn.Parameter(torch.empty(numeric_count, token_dim))
        nn.init.normal_(self.numeric_weight, std=0.02)
        nn.init.normal_(self.numeric_bias, std=0.02)
        self.cls_token = nn.Parameter(torch.empty(1, 1, token_dim))
        nn.init.normal_(self.cls_token, std=0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=token_dim, nhead=4, dim_feedforward=token_dim * 4, dropout=0.20,
            activation="gelu", batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=1)
        self.head = nn.Sequential(
            nn.LayerNorm(token_dim), nn.Linear(token_dim, 64), nn.GELU(), nn.Dropout(0.20),
            nn.Linear(64, 1),
        )

    def forward(self, categorical, numeric):
        cat_tokens = torch.stack(
            [embedding(categorical[:, index]) for index, embedding in enumerate(self.categorical_embeddings)], dim=1
        )
        numeric_tokens = numeric.unsqueeze(-1) * self.numeric_weight.unsqueeze(0) + self.numeric_bias.unsqueeze(0)
        cls = self.cls_token.expand(categorical.shape[0], -1, -1)
        encoded = self.transformer(torch.cat([cls, cat_tokens, numeric_tokens], dim=1))
        return self.head(encoded[:, 0]).squeeze(1)


def predict(model, categorical, numeric, mean, std):
    model.eval()
    loader = DataLoader(TensorDataset(torch.from_numpy(categorical), torch.from_numpy(numeric)), batch_size=BATCH_SIZE)
    outputs = []
    with torch.inference_mode():
        for categorical_batch, numeric_batch in loader:
            outputs.append(model(categorical_batch, numeric_batch).cpu())
    return np.expm1(torch.cat(outputs).numpy() * std + mean).clip(min=0)


_, datasets = load_datasets(ROOT / "data")
raw_train = datasets["train"].copy()
raw_train["period"] = pd.to_datetime(raw_train["RENT_APPROVAL_DATE"] + "-01").dt.to_period("M")
periods = np.sort(raw_train["period"].unique())
cutoff = periods[int(len(periods) * 0.80)]
raw_fit = raw_train.loc[raw_train["period"] < cutoff].copy()
raw_valid = raw_train.loc[raw_train["period"] >= cutoff].copy()
preprocessor = RentalPreprocessor(datasets)
fit_base = base.preprocess_partition(preprocessor, raw_fit, fit=True)
valid_base = base.preprocess_partition(preprocessor, raw_valid)
target_transformer = base.TargetFeatureTransformer(base.LAG_MONTHS)
fit_targets = target_transformer.fit_transform_train(base.labelled_history(raw_fit, fit_base))
valid_targets = target_transformer.transform(base.labelled_history(raw_valid, valid_base))
fit_frame = base.model_frame(fit_base, fit_targets, datasets)
valid_frame = base.model_frame(valid_base, valid_targets, datasets)
encoder = base.TabularEncoder(base.CATEGORICAL_FEATURES, base.NUMERIC_FEATURES).fit(fit_frame)
fit_cat, fit_num = encoder.transform(fit_frame)
valid_cat, valid_num = encoder.transform(valid_frame)
y_fit = raw_fit[base.TARGET].to_numpy(dtype=np.float32)
y_valid = raw_valid[base.TARGET].to_numpy(dtype=np.float32)
log_fit = np.log1p(y_fit)
mean, std = float(log_fit.mean()), float(log_fit.std())
y_scaled = (log_fit - mean) / std

model = FTTransformerRegressor(encoder.cardinalities, len(base.NUMERIC_FEATURES))
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0002, weight_decay=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)
loss_function = nn.HuberLoss(delta=1.0)
loader = DataLoader(TensorDataset(torch.from_numpy(fit_cat), torch.from_numpy(fit_num), torch.from_numpy(y_scaled)),
                    batch_size=BATCH_SIZE, shuffle=True, generator=torch.Generator().manual_seed(SEED))

best_state, best_rmse, best_epoch, wait = None, float("inf"), 0, PATIENCE
history = []
print(f"FT-Transformer DL | fit={len(raw_fit):,}, valid={len(raw_valid):,}, cutoff={cutoff}")
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for categorical_batch, numeric_batch, target_batch in loader:
        optimizer.zero_grad(set_to_none=True)
        loss = loss_function(model(categorical_batch, numeric_batch), target_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * len(target_batch)
    valid_prediction = predict(model, valid_cat, valid_num, mean, std)
    rmse = mean_squared_error(y_valid, valid_prediction) ** 0.5
    scheduler.step(rmse)
    history.append({"epoch": epoch, "train_huber": total_loss / len(raw_fit), "validation_rmse": rmse})
    print(f"epoch {epoch:02d}: train Huber={total_loss / len(raw_fit):.5f}, validation RMSE={rmse:.3f}")
    if rmse < best_rmse:
        best_rmse, best_epoch, wait = rmse, epoch, PATIENCE
        best_state = {name: value.detach().clone() for name, value in model.state_dict().items()}
    else:
        wait -= 1
        if wait == 0:
            print(f"Early stopping; best epoch={best_epoch}.")
            break

model.load_state_dict(best_state)
prediction = predict(model, valid_cat, valid_num, mean, std)
metrics = {
    "experiment": "exp_10_ft_transformer", "framework": "PyTorch", "tree_models_used": False,
    "model": "FT-Transformer with numeric and categorical feature tokenization",
    "validation_protocol": "chronological: train before 2024-05; validate from 2024-05 onward",
    "fit_rows": len(raw_fit), "validation_rows": len(raw_valid), "best_epoch": best_epoch,
    "RMSE": float(mean_squared_error(y_valid, prediction) ** 0.5),
    "MAE": float(mean_absolute_error(y_valid, prediction)), "R2": float(r2_score(y_valid, prediction)),
}
output_dir = ROOT / "exploration" / "metul"
(output_dir / "exp_10_ft_transformer_metrics.json").write_text(json.dumps(metrics, indent=2))
pd.DataFrame(history).to_csv(output_dir / "exp_10_ft_transformer_history.csv", index=False)
print(pd.Series(metrics)[["RMSE", "MAE", "R2", "best_epoch"]].to_string())


## Recorded result

At the 30-epoch limit, validation was still improving; the final/best checkpoint produced **RMSE 497.497**, **MAE 361.312**, and **R² 0.4266**. It improves on the explicit trend-residual variant (518.590), but not the initial TabTransformer baseline (495.853).

In [ ]:
from pathlib import Path
import json
import pandas as pd

metrics_path = Path('exp_10_ft_transformer_metrics.json')
if not metrics_path.exists():
    metrics_path = Path('exploration/metul/exp_10_ft_transformer_metrics.json')
metrics = json.loads(metrics_path.read_text())
pd.Series(metrics)[['RMSE', 'MAE', 'R2', 'best_epoch', 'fit_rows', 'validation_rows']]